In [ ]:
import cafe
import pandas as pd

# cafe.logger.setLevel("DEBUG")

extract from function

In [ ]:
method_df = cafe.method.scan_method(return_type="dataframe")

# only scan these methods
method_list = [
    # velocity
    # velocity_mst
    "scvelo",
    "dynamo",
    "velovi",
    "pyrovelocity",
    "veloae",
    "celldancer",
    "unitvelo",
    
    # pseudotime
    "comp1",
    "palantir",
    "cytotrace2",
    "sctc",

    # cluster
    "cluster_mst",
    "stavia",
    
    # lineage
    "cellrank",
]

method_df = method_df.loc[method_list] # only keep these methods
method_df.to_csv("methods.csv")
method_df

,name,version,description,wrapper_type,doi,github_url,use_gpu,cpu_parallelization,parameter
scvelo,scvelo,0.0.1,scVelo: RNA velocity generalized through dynam...,velocity,10.1038/s41587-020-0591-3,https://github.com/theislab/scvelo,False,True,"{'adata': {'name': 'adata', 'required': True, ..."
dynamo,dynamo,0.0.1,Dynamo: Mapping Transcriptomic Vector Fields o...,velocity,10.1016/j.cell.2021.12.045,https://github.com/aristoteleo/dynamo-release,False,True,"{'adata': {'name': 'adata', 'required': True, ..."
velovi,velovi,0.0.1,VeloVI: Deep generative modeling of transcript...,velocity,10.1038/s41592-023-01994-w,https://github.com/yoseflab/velovi,NaN,NaN,"{'adata': {'name': 'adata', 'required': True, ..."
pyrovelocity,pyrovelocity,0.0.1,PyroVelocity: probabilistic modeling of RNA ve...,velocity,10.1101/2022.09.12.507691,https://github.com/pinellolab/pyrovelocity,NaN,NaN,"{'adata': {'name': 'adata', 'required': True, ..."
veloae,veloae,0.0.1,VeloAE: Representation learning of RNA velocit...,velocity,10.1073/pnas.2105859118,https://github.com/qiaochen/VeloAE,NaN,NaN,"{'adata': {'name': 'adata', 'required': True, ..."
celldancer,celldancer,0.0.1,CellDancer: Estimating Cell-dependent RNA Velo...,velocity,10.1038/s41587-023-01728-5,https://github.com/GuangyuWangLab2021/cellDancer,NaN,NaN,"{'adata': {'name': 'adata', 'required': True, ..."
unitvelo,unitvelo,0.0.1,UniTVelo: temporally unified RNA velocity rein...,velocity,10.1038/s41467-022-34188-7,https://github.com/StatBiomed/UniTVelo,NaN,NaN,"{'adata': {'name': 'adata', 'required': True, ..."
comp1,comp1,0.0.1,"Comp1: baseline for linear wrapper, extract an...",linear,NaN,NaN,NaN,NaN,"{'adata': {'name': 'adata', 'required': True, ..."
palantir,palantir,0.0.1,Palantir: characterization of cell fate probab...,"[linear, probability, lineage]",10.1038/s41587-019-0068-4,https://github.com/dpeerlab/Palantir,NaN,NaN,"{'adata': {'name': 'adata', 'required': True, ..."
cytotrace2,cytotrace2,0.0.1,Cytotrace2: cellular potency categories and ab...,linear,10.1101/2024.03.19.585637,https://github.com/digitalcytometry/cytotrace2,NaN,NaN,"{'adata': {'name': 'adata', 'required': True, ..."


In [ ]:
# only focus on specific prior knowledge parameters
prior_keys = ["cluster", "basis", "start_cell"]

# construct prior information dataframe
prior_dict = {}
for method, param_dict in method_df["parameter"].items():
    method_prior_dict = {}
    for k in prior_keys:
        if k not in param_dict:
            method_prior_dict[k] = "Not Required"
        else:
            if param_dict[k]["required"]:
                method_prior_dict[k] = "Necessary"
            else:
                method_prior_dict[k] = "Optional" # not required but recommended, which make reuslt more accurate.
    prior_dict[method] = method_prior_dict
prior_information_df = pd.DataFrame.from_dict(prior_dict, orient="index", columns=prior_keys)

prior_information_df

,cluster,basis,start_cell
scvelo,Not Required,Not Required,Not Required
dynamo,Not Required,Necessary,Not Required
velovi,Not Required,Not Required,Not Required
pyrovelocity,Not Required,Not Required,Not Required
veloae,Not Required,Not Required,Not Required
celldancer,Necessary,Necessary,Not Required
unitvelo,Necessary,Not Required,Not Required
comp1,Not Required,Optional,Not Required
palantir,Optional,Not Required,Necessary
cytotrace2,Optional,Not Required,Not Required


extract from dynverse docker

In [ ]:
dynverse_method_list = ["ti_slingshot", "ti_paga", "ti_mst"]
# dynverse_method_list = ["ti_slingshot"]
dynverse_p2p = {
    "dimred": "basis"
} # dynverse_paramter_transfer_dict
dynverse_pi2i = {
    "groups_id": "cluster", 
    "start_id": "start_cell"
} # dynverse_prior_information_transfer_dict
dynverse_prior_dict = {}    

for method_name in dynverse_method_list:
    # extract dynverse method input dataframe
    method = cafe.method.FateMethod(method_name=method_name, backend_name="dynverse")
    method.choose_backend()
    inputs_df = method.method_backend.definition.get_inputs_df()
    method_prior_dict = {}
    # extract all prior_information rows
    pi_inputs_df = inputs_df[inputs_df["type"] == "prior_information"]
    pi_inputs_df["input_id"] = pi_inputs_df["input_id"].apply(lambda x: dynverse_pi2i[x] if x in dynverse_pi2i else x)
    # extract specific paramter rows 
    p_inputs_df = inputs_df[inputs_df["type"] == "parameter"].query(f"input_id in {list(dynverse_p2p.keys())}")
    p_inputs_df["input_id"] = p_inputs_df["input_id"].map(dynverse_p2p)
    # transfter to "Necessary"/"Optional"
    sub_input_df = pd.concat([pi_inputs_df, p_inputs_df], axis=0)
    sub_input_df.set_index("input_id", inplace=True)
    dynverse_prior_dict[method_name] = sub_input_df.apply(lambda row: "Necessary" if row["required"] else "Optional", axis=1).to_dict()

dynverse_prior_information_df = pd.DataFrame.from_dict(dynverse_prior_dict, orient="index").fillna("Not Required")
dynverse_prior_information_df

INFO     |- backend:'conda' is not available for method:'ti_slingshot', choosing new backend: 'dynverse_docker'                                                                                         
INFO     |- method backend loaded: Dynverse Docker Backend: docker image 'dynverse/ti_slingshot:v1.0.3'                                                                                                 
INFO     |- backend:'conda' is not available for method:'ti_paga', choosing new backend: 'dynverse_docker'                                                                                              
INFO     |- method backend loaded: Dynverse Docker Backend: docker image 'dynverse/ti_paga:v0.9.9.05'                                                                                                   
INFO     |- backend:'conda' is not available for method:'ti_mst', choosing new backend: 'dynverse_docker'                                                                                           

,start_cell,end_id,cluster,basis
ti_slingshot,Optional,Optional,Not Required,Not Required
ti_paga,Necessary,Not Required,Optional,Not Required
ti_mst,Not Required,Not Required,Not Required,Optional


In [ ]:
prior_information_df = pd.concat([prior_information_df, dynverse_prior_information_df], axis=0)
prior_information_df = prior_information_df.fillna("Not Required")

# colorful visualization
color_map = {
    "Not Required": "background-color: green; color: white; font-weight: bold;",
    "Necessary": "background-color: red; color: white; font-weight: bold;",
    "Optional": "background-color: blue; color: white; font-weight: bold;"
}
def color_cells(val):
    return color_map.get(val, "")

styled_df = prior_information_df.style.applymap(color_cells)
display(styled_df)


,cluster,basis,start_cell,end_id
scvelo,Not Required,Not Required,Not Required,Not Required
dynamo,Not Required,Necessary,Not Required,Not Required
velovi,Not Required,Not Required,Not Required,Not Required
pyrovelocity,Not Required,Not Required,Not Required,Not Required
veloae,Not Required,Not Required,Not Required,Not Required
celldancer,Necessary,Necessary,Not Required,Not Required
unitvelo,Necessary,Not Required,Not Required,Not Required
comp1,Not Required,Optional,Not Required,Not Required
palantir,Optional,Not Required,Necessary,Not Required
cytotrace2,Optional,Not Required,Not Required,Not Required


auto select approprate methods based on data prior information

In [ ]:
def check_available_method_list(fadata, prior_information_df):
    """
    Return a list of methods that can be used for the given fadata,
    based on whether all 'Necessary' prior knowledge is present in fadata.prior_information.
    """
    available_method_list = []
    # 获取数据的先验信息key集合
    data_priors = set(fadata.prior_information.keys())
    for method, row in prior_information_df.iterrows():
        ok = True
        for prior, need in row.items():
            if need == "Necessary" and prior not in data_priors:
                ok = False
                break
        if ok:
            available_method_list.append(method)
    return available_method_list

In [ ]:
# pancreas: 所有方法都可用
fadata = cafe.data.read_pancreas()
print(fadata.prior_information)
check_available_method_list(fadata, prior_information_df)

{'cluster': 'clusters', 'basis': 'X_umap', 'start_cell': 'cell_1103'}


['scvelo',
 'dynamo',
 'velovi',
 'pyrovelocity',
 'veloae',
 'celldancer',
 'unitvelo',
 'comp1',
 'palantir',
 'cytotrace2',
 'sctc',
 'cluster_mst',
 'stavia',
 'cellrank',
 'ti_slingshot',
 'ti_paga',
 'ti_mst']

In [ ]:
# bonemarrow: 暂时缺少了start_cell, 不能用stavia和palantir``
fadata2 = cafe.data.read_bonemarrow()
print(fadata2.prior_information)
check_available_method_list(fadata2, prior_information_df)

{'cluster': 'clusters', 'basis': 'X_tsne'}


['scvelo',
 'dynamo',
 'velovi',
 'pyrovelocity',
 'veloae',
 'celldancer',
 'unitvelo',
 'comp1',
 'cytotrace2',
 'sctc',
 'cluster_mst',
 'cellrank',
 'ti_slingshot',
 'ti_mst']

划分为先验知识level：free，weakly restricted， strong restricted
- free: 完全不提供先验知识的，所有都是Not Required的
- weakly restricted: 不强制提供先验知识，只是可选的，所有都是Not Required的或者Optional的。
- strong restricted: 必须提供先验知识，存在Necessary。

In [ ]:
# 按照先验知识level进行划分
def classify_prior_level(row):
    # free: 全部Not Required
    if all(val == "Not Required" for val in row):
        return "free"
    # strong restricted: 有Necessary
    if any(val == "Necessary" for val in row):
        return "strong restricted"
    # weakly restricted: 有Optional但无Necessary
    if any(val == "Optional" for val in row):
        return "weakly restricted"
    return "unknown"
prior_information_df_with_level = prior_information_df.copy()
prior_information_df_with_level["level"] = prior_information_df_with_level.apply(classify_prior_level, axis=1)

prior_information_df_with_level.to_csv("prior_information.csv")

# colorful visualization
color_map = {
    "Not Required": "background-color: green; color: white; font-weight: bold;",
    "Necessary": "background-color: red; color: white; font-weight: bold;",
    "Optional": "background-color: blue; color: white; font-weight: bold;",
    "free": "background-color: lightgreen; color: black; font-weight: bold;",
    "weakly restricted": "background-color: lightblue; color: black; font-weight: bold;",
    "strong restricted": "background-color: orange; color: black; font-weight: bold;"
}
def color_cells(val):
    return color_map.get(val, "")

styled_df = prior_information_df_with_level.style.applymap(color_cells)
display(styled_df)

,cluster,basis,start_cell,end_id,level
scvelo,Not Required,Not Required,Not Required,Not Required,free
dynamo,Not Required,Necessary,Not Required,Not Required,strong restricted
velovi,Not Required,Not Required,Not Required,Not Required,free
pyrovelocity,Not Required,Not Required,Not Required,Not Required,free
veloae,Not Required,Not Required,Not Required,Not Required,free
celldancer,Necessary,Necessary,Not Required,Not Required,strong restricted
unitvelo,Necessary,Not Required,Not Required,Not Required,strong restricted
comp1,Not Required,Optional,Not Required,Not Required,weakly restricted
palantir,Optional,Not Required,Necessary,Not Required,strong restricted
cytotrace2,Optional,Not Required,Not Required,Not Required,weakly restricted
